In [1]:
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split
import pandas as pd
import pickle

In [2]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [3]:
## here Estimated salary is continuos figure and so this problem statement is a Regression problem statement

## Preprocess the data
# Drop irrelevant features
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1) # axis=1 for columns

data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0


In [4]:
# here Geography and Gender are categorical features, we need to encode them

# Encode categorical features
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender']) # this will convert Male to 1 and Female to 0



data.head()

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,0,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,0,41,1,83807.86,1,0,1,112542.58,0
2,502,France,0,42,8,159660.80,3,1,0,113931.57,1
3,699,France,0,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,0,43,2,125510.82,1,1,1,79084.10,0


In [6]:
# label_encoder_geography = LabelEncoder()
# data['Geography'] = label_encoder_geography.fit_transform(data['Geography']) # this will convert France to 0, Germany to 1 and Spain to 2

# here we will not use label encoding for Geography because it will create an ordinal relationship between the categories, which is not true. Instead, we will use one-hot encoding.
# data = pd.get_dummies(data, columns=['Geography'], prefix='Geography')

from sklearn.preprocessing import OneHotEncoder
one_hot_encoder_geo = OneHotEncoder() 
geo_encoder = one_hot_encoder_geo.fit_transform(data[['Geography']]) # fit and transform the Geography column
geo_encoded_df = pd.DataFrame(geo_encoder.toarray(), columns=one_hot_encoder_geo.get_feature_names_out(['Geography']))

geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0
...,...,...,...
9995,1.0,0.0,0.0
9996,1.0,0.0,0.0
9997,1.0,0.0,0.0
9998,0.0,1.0,0.0


In [7]:
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [8]:
## divide the dataset into independent and dependent features

x = data.drop('EstimatedSalary',axis = 1)
y = data['EstimatedSalary']

In [9]:
## split data in training and testing sets
X_train, X_test, y_train, y_test = train_test_split(x,y, test_size=0.2, random_state=42)

## Scale the features
scalar = StandardScaler()
X_train = scalar.fit_transform(X_train)
X_test = scalar.fit_transform(X_test)

In [10]:
#saving encoders and scalar - one_hot_encoder_geo and label_encoder_gender as pickle file - file in disk, so that we can use this in E2E project

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('one_hot_encoder_geo.pkl','wb') as file:
    pickle.dump(one_hot_encoder_geo,file)

# saving the scalar of X_train and X_test in pickle format

with open('scalar.pkl','wb') as file:
    pickle.dump(scalar,file)

## Train ANN Regression Problem Statement

In [11]:
## ANN Implementation

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [12]:
## Build our ANN model

## give input and hidden layer 1 with 64 neurons
model = Sequential([
    Dense(64, activation='relu',input_shape=(X_train.shape[1],)), # HL1 connected with input layer
    Dense(32, activation='relu'), # HL2
    Dense(1) # output layer for regression, if no activation fn is given, by default would take linear activation fn
]
)

In [14]:
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

#search for keras loss functions in google, to get to know what all loss functions are available for both classification and regression
#problem statements

In [15]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 64)                832       
                                                                 
 dense_1 (Dense)             (None, 32)                2080      
                                                                 
 dense_2 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [17]:
# Setup the tensorboard so that we can visualize the training logs
import datetime
from  tensorflow.keras.callbacks import EarlyStopping,TensorBoard

log_dir = "regressionlogs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback = TensorBoard(log_dir=log_dir,histogram_freq=1)

In [18]:
early_stopping_callback = EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True) 
# validation loss and patience = 5 means wait for atleast 5 epochs 

In [19]:
### Training the model

history = model.fit(
    X_train,y_train,
    validation_data = (X_test,y_test),
    epochs = 100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100


250/250 [==============================] - 3s 5ms/step - loss: 100373.1406 - mae: 100373.1406 - val_loss: 98500.8828 - val_mae: 98500.8828
Epoch 2/100
250/250 [==============================] - 1s 3ms/step - loss: 99579.5859 - mae: 99579.5859 - val_loss: 96909.6172 - val_mae: 96909.6172
Epoch 3/100
250/250 [==============================] - 1s 3ms/step - loss: 96828.9531 - mae: 96828.9531 - val_loss: 92894.5078 - val_mae: 92894.5078
Epoch 4/100
250/250 [==============================] - 1s 4ms/step - loss: 91476.3047 - mae: 91476.3047 - val_loss: 86270.1484 - val_mae: 86270.1484
Epoch 5/100
250/250 [==============================] - 1s 3ms/step - loss: 83734.8516 - mae: 83734.8516 - val_loss: 77812.6172 - val_mae: 77812.6172
Epoch 6/100
250/250 [==============================] - 1s 4ms/step - loss: 74710.5938 - mae: 74710.5938 - val_loss: 68982.3828 - val_mae: 68982.3828
Epoch 7/100
250/250 [==============================] - 1s 3ms/step - loss: 65914.8984 - mae: 65914.898

In [20]:
%load_ext tensorboard

In [22]:

%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 16372), started 0:00:39 ago. (Use '!kill 16372' to kill it.)

In [23]:
## Evaluate model on test data

test_loss, test_mae = model.evaluate(X_test,y_test)
print(f'Test MAE :  {test_mae}')

63/63 [==============================] - 0s 1ms/step - loss: 50255.4961 - mae: 50255.4961
Test MAE :  50255.49609375


In [24]:
model.save('regression_model.h5')

c:\Users\bharg\myPython\secondSampleProj\myenv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
